# Phase 2 — Data Engineering Check
Unpivot the wide sales data into long format and join with calendar + prices.

In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"D:\Retail Demand Forecasting\.venv\Scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"D:\Retail Demand Forecasting\.venv\Scripts\python.exe"
os.environ["SPARK_LOCAL_DIRS"] = r"D:\spark-temp"
os.environ["HADOOP_HOME"] = r"D:\hadoop"

os.chdir(r"D:\Retail Demand Forecasting")

In [2]:
from pyspark.sql import SparkSession
from retail_demand_forecasting.nodes.data_engineering import unpivot_sales

os.makedirs(r"D:\spark-temp", exist_ok=True)

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("phase2_data_engineering")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.local.dir", r"D:\spark-temp")
    .getOrCreate()
)
print(f"Spark {spark.version} ready.")

Spark 4.1.1 ready.


## 1. Load raw datasets

In [3]:
PROJECT_ROOT = r"D:\Retail Demand Forecasting"

sales_train_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(os.path.join(PROJECT_ROOT, "data/01_raw/sales_train_validation.csv"))
)
calendar_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(os.path.join(PROJECT_ROOT, "data/01_raw/calendar.csv"))
)
sell_prices_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(os.path.join(PROJECT_ROOT, "data/01_raw/sell_prices.csv"))
)
print(f"sales_train: {sales_train_raw.count():,} rows, {len(sales_train_raw.columns)} cols")
print(f"calendar:    {calendar_raw.count():,} rows, {len(calendar_raw.columns)} cols")
print(f"sell_prices: {sell_prices_raw.count():,} rows, {len(sell_prices_raw.columns)} cols")

sales_train: 30,490 rows, 1919 cols
calendar:    1,969 rows, 14 cols
sell_prices: 6,841,121 rows, 4 cols


## 2. Run unpivot_sales node

In [ ]:
melted_df = unpivot_sales(sales_train_raw, calendar_raw, sell_prices_raw)

## 3. Schema & sample rows

In [ ]:
melted_df.printSchema()

In [ ]:
melted_df.limit(10).toPandas()

## 4. Row count

In [ ]:
print(f"Total rows: {melted_df.count():,}")

## 5. Save to intermediate parquet

In [ ]:
output_path = os.path.join(PROJECT_ROOT, "data/02_intermediate/sales_melted.parquet")
melted_df.write.mode("overwrite").parquet(output_path)
print(f"Saved to {output_path}")

In [ ]:
spark.stop()
print("Phase 2 complete.")